# M4 - Fine-tune GEETA 4B (LoRA text + vision tower) -> merge -> GGUF -> serve smoke

Implements PROJECT_BRIEF C3 (fine-tune), C4 (serve) and R1 (verify the new Gated-DeltaNet + vision path early).

| Cell | What | Gate | Resume flag |
|------|------|------|-------------|
| C1 | setup: paths, resume helpers, shared dataset + LoRA runner | - | - |
| A | asset verify (key layout, token ids, template alignment, data) | assertions | `A` |
| B | OCR smoke on BASE 8-bit (2 real + 3 synth pages) | all non-empty | `B` |
| C | text-LoRA overfit check (20 prefix-expanded dialogues) | final loss < 0.9 x first | `C` |
| D | full text LoRA (all 706 dialogues, one epoch, grad accum) | val loss decreases | `D` |
| E | vision-tower fine-tune (60 rows, custom loop) | loss drops | `E` |
| F | merge LoRA + tower into standalone MLX dir + round-trip verify | mlx_vlm loads + generates | `F` |
| G | HF dir assembly + GGUF (f16 + Q4_K_M) | arch + tensor counts | `G` |
| H | llama-server smoke (text + vision, TTFT) with MLX fallback | 200 + sensible text | `H` |
| I | summary | - | `I` |

**Resume semantics:** each code cell checks `outputs/m4/.<flag>.done` first and no-ops when present.
Re-running the notebook is safe. Run from the **project root**:

    nohup .venv/bin/jupyter execute --kernel python3 notebooks/04_finetune.ipynb > logs/m4_finetune.log 2>&1 &

Hardware: Apple M4 Max, 48 GB unified memory. Base: `models/Qwen3.5-4B-MLX-8bit`
(5.16 GB: LM 8-bit quant + vision tower bf16). Disk budget: merged MLX dir ~9.5 GB,
f16 GGUF ~9.6 GB, Q4_K_M ~3.5 GB.


In [ ]:

# ---- C1 setup: paths, resume helpers, shared constants + classes -----------
import json, os, sys, time, types, collections, traceback, hashlib
from pathlib import Path

import mlx.core as mx
from mlx.utils import tree_flatten, tree_unflatten

# jupyter execute may set the kernel cwd to the notebook dir; resolve root robustly.
ROOT = Path.cwd()
if not (ROOT / "PROJECT_BRIEF.md").exists():
    for cand in (ROOT, ROOT.parent, ROOT.parent.parent):
        if (cand / "PROJECT_BRIEF.md").exists():
            ROOT = cand
            break
os.chdir(ROOT)
assert (ROOT / "PROJECT_BRIEF.md").exists(), "cannot locate PROJECT_BRIEF.md from cwd=" + str(Path.cwd())

MLX_DIR = "models/Qwen3.5-4B-MLX-8bit"
HF_DIR = "models/Qwen3.5-4B"
GEETA_MLX_DIR = "models/geeta-4b-mlx"
GEETA_HF_DIR = "models/geeta-4b-hf"
GEETA_GGUF_DIR = "models/geeta-4b"
LLAMA_CPP = "third_party/llama.cpp"
OUT = ROOT / "outputs" / "m4"
OUT.mkdir(parents=True, exist_ok=True)

SENT = chr(10) + "image" + chr(10)              # 7-char chat-template sentinel
ASSIST_OPEN = chr(96) + "s" + chr(105) + chr(115) + chr(116) + "ant"
SYS_OPEN = chr(115) + chr(121) + "stem"

def done(flag):
    return (OUT / ("." + flag + ".done")).exists()

def mark(flag, payload=None):
    p = OUT / ("." + flag + ".done")
    if not p.exists():
        p.write_text(json.dumps({"flag": flag, "ts": time.time(), "payload": payload}) + chr(10))
    print("[m4] flagged", flag, "done")

def log(name, payload):
    path = OUT / name
    path.write_text(json.dumps(payload, indent=2, default=str))
    print("[m4] wrote", path)

def mem():
    return "peak=" + str(round(mx.get_peak_memory() / 2**30, 1)) + "GiB"

# --- shared chat dataset: exact per-row prefix/offset, enable_thinking=False --
class GeetaChatDataset:
    def __init__(self, rows, tokenizer, max_seq_length=2048):
        self.tokenizer = tokenizer
        self.max_seq_length = max_seq_length
        self.rows = rows
    def process(self, row):
        messages = row["messages"]
        agp = messages[-1]["role"] == "user"
        tokens = self.tokenizer.apply_chat_template(messages, tools=None,
            add_generation_prompt=agp, enable_thinking=False, return_dict=False)
        if len(messages) > 1:
            prefix = self.tokenizer.apply_chat_template(messages[:-1], tools=None,
                add_generation_prompt=True, enable_thinking=False, return_dict=False)
        else:
            prefix = []
        return list(tokens), len(prefix)
    def __getitem__(self, idx):
        return self.rows[idx]
    def __len__(self):
        return len(self.rows)

def expand(rows):
    """Prefix-expand each dialogue into one row per assistant turn (loss = that turn)."""
    out = []
    for r in rows:
        msgs = r["messages"]
        n_asst = sum(1 for m in msgs if m["role"] == "assistant")
        for i in range(n_asst):
            out.append({"messages": msgs[: 2 * i + 2], "source": r.get("source"),
                        "role": r.get("role"), "expansion_of": r.get("id")})
    return out

class LossCapture:
    def __init__(self):
        self.train = []
        self.val = []
    def on_train_loss_report(self, info):
        self.train.append({"it": info["iteration"], "loss": round(info["train_loss"], 4),
                           "lr": info["learning_rate"], "tok_s": round(info["tokens_per_second"], 1)})
    def on_val_loss_report(self, info):
        self.val.append({"it": info["iteration"], "loss": round(info["val_loss"], 4)})

def run_lora_training(model, tokenizer, train_rows, valid_rows, *,
                      num_layers=16, lora_params=None, batch_size=1, iters=100,
                      val_batches=5, learning_rate=1e-5, lr_schedule=None,
                      max_seq_length=2048, grad_checkpoint=True, grad_accum=1,
                      steps_per_report=10, steps_per_eval=50, save_every=None,
                      adapter_path="outputs/m4/lora", seed=0):
    """Reproduce mlx_lm.lora.train_model() plumbing with our exact-offset datasets."""
    import mlx.nn as nn
    import mlx.optimizers as optim
    from mlx_lm.tuner.datasets import CacheDataset
    from mlx_lm.tuner.trainer import TrainingArgs, train
    from mlx_lm.tuner.utils import linear_to_lora_layers, build_schedule
    from mlx_lm.utils import save_config

    if lora_params is None:
        lora_params = {"rank": 16, "scale": 2.0, "dropout": 0.0}
    adapter_path = Path(adapter_path)
    adapter_path.mkdir(parents=True, exist_ok=True)

    model.freeze()
    linear_to_lora_layers(model, num_layers, lora_params)
    n_train = sum(p.size for _, p in tree_flatten(model.trainable_parameters()))
    n_total = sum(p.size for _, p in tree_flatten(model.parameters()))
    print("[lora] trainable %.1fM / %.1fM (%.2f%%)" % (n_train/1e6, n_total/1e6, 100*n_train/max(n_total,1)))

    args_ns = types.SimpleNamespace(
        model=MLX_DIR, train=True, fine_tune_type="lora", optimizer="adam",
        data=str(adapter_path), seed=seed, num_layers=num_layers, batch_size=batch_size,
        iters=iters, val_batches=val_batches, learning_rate=learning_rate,
        steps_per_report=steps_per_report, steps_per_eval=steps_per_eval,
        resume_adapter_file=None, adapter_path=str(adapter_path), save_every=save_every or iters,
        test=False, test_batches=500, max_seq_length=max_seq_length, config=None,
        grad_checkpoint=grad_checkpoint, grad_accumulation_steps=grad_accum,
        clear_cache_threshold=0, lr_schedule=lr_schedule, lora_parameters=lora_params,
        mask_prompt=True, report_to=None, project_name="geeta")
    save_config(vars(args_ns), adapter_path / "adapter_config.json")

    training_args = TrainingArgs(
        batch_size=batch_size, iters=iters, val_batches=val_batches,
        steps_per_report=steps_per_report, steps_per_eval=steps_per_eval,
        steps_per_save=save_every or iters, adapter_file=str(adapter_path / "adapters.safetensors"),
        max_seq_length=max_seq_length, grad_checkpoint=grad_checkpoint,
        grad_accumulation_steps=grad_accum)

    lr = build_schedule(lr_schedule) if lr_schedule else learning_rate
    opt = optim.Adam(learning_rate=lr)

    train_set = GeetaChatDataset(train_rows, tokenizer, max_seq_length=max_seq_length)
    valid_set = GeetaChatDataset(valid_rows, tokenizer, max_seq_length=max_seq_length) if valid_rows else None

    cb = LossCapture()
    train(model=model, args=training_args, optimizer=opt,
          train_dataset=CacheDataset(train_set),
          val_dataset=CacheDataset(valid_set) if valid_set else None,
          training_callback=cb)

    mx.save_safetensors(str(adapter_path / "adapters.safetensors"),
                        dict(tree_flatten(model.trainable_parameters())))
    return cb

print("C1 ready | root:", str(ROOT))


In [ ]:

# ---- Cell A: asset verification --------------------------------------------
if done("A"):
    print("A already done - skipping")
else:
    import mlx_lm
    from safetensors import safe_open

    rep = {"mlx_dir": MLX_DIR, "checks": {}}

    # 1) raw key layout of the 8-bit MLX dir
    with safe_open(os.path.join(MLX_DIR, "model.safetensors"), framework="np") as f:
        keys = list(f.keys())
        top = collections.Counter(k.split(".")[0] for k in keys)
    rep["checks"]["dir_keys_total"] = len(keys)
    rep["checks"]["dir_top_prefixes"] = dict(top)
    layout_ok = (len(keys) == 1221 and top.get("language_model") == 924
                 and top.get("vision_tower") == 297)
    assert layout_ok, "unexpected key layout: %d %s" % (len(keys), dict(top))

    # 2) mlx_lm model load (lazy: structure only, then eval) + config
    model_lm, tok, cfg = mlx_lm.load(MLX_DIR, lazy=True, return_config=True)
    mx.eval(model_lm)
    pkeys = [k for k, _ in tree_flatten(model_lm.parameters())]
    ptop = collections.Counter(k.split(".")[0] for k in pkeys)
    rep["checks"]["model_param_keys"] = len(pkeys)
    rep["checks"]["model_top_prefixes"] = dict(ptop)
    rep["checks"]["model_layers"] = len(model_lm.layers)
    assert len(pkeys) == 924 and ptop.get("language_model") == 924
    assert len(model_lm.layers) == 32

    # 3) config token ids (top-level, falling back to text_config)
    tc = cfg.get("text_config", {}) or {}
    def cid(k):
        v = cfg.get(k)
        return v if v is not None else tc.get(k)
    rep["checks"]["config_token_ids"] = {k: cid(k) for k in
        ("image_token_id", "video_token_id", "vision_start_token_id", "vision_end_token_id",
         "assistant_token_id", "eos_token_id")}
    assert cid("image_token_id") == 248056, cid("image_token_id")
    rep["checks"]["vocab_size"] = cid("vocab_size") or 248320

    # 4) tokenizer sentinels
    rep["checks"]["sentinel_token_ids"] = [int(x) for x in tok.encode(SENT)]
    rep["checks"]["assistant_token_id"] = tok.convert_tokens_to_ids(ASSIST_OPEN)

    # 5) chat template present + data inventory
    rep["checks"]["chat_template_present"] = os.path.exists(os.path.join(MLX_DIR, "chat_template.jinja"))
    n_train = sum(1 for _ in open("data/splits/train.jsonl"))
    n_vision = sum(1 for _ in open("data/splits/train_vision.jsonl"))
    rep["checks"]["train_rows"] = n_train
    rep["checks"]["vision_rows"] = n_vision
    rep["checks"]["heldout_rows"] = sum(1 for _ in open("data/eval/heldout_problems.jsonl"))
    rep["checks"]["probe_rows"] = sum(1 for _ in open("data/eval/adversarial_probes.jsonl"))
    assert n_train == 706 and n_vision == 60

    # 6) template alignment over ALL rows with enable_thinking=False (the crux)
    def render(msgs, agp):
        return tok.apply_chat_template(msgs, tools=None, add_generation_prompt=agp,
                                       enable_thinking=False, return_dict=False)
    trows = [json.loads(l) for l in open("data/splits/train.jsonl")]
    t_mis = 0; t_max = 0
    for r in trows:
        m = r["messages"]
        full = render(m, False)
        pre = render(m[:-1], True) if len(m) > 1 else []
        if list(full[:len(pre)]) != list(pre):
            t_mis += 1
        t_max = max(t_max, len(full))
    vrows = [json.loads(l) for l in open("data/splits/train_vision.jsonl")]
    v_mis = 0; v_max = 0
    for r in vrows:
        m = r["messages"]
        full = render(m, False)
        pre = render(m[:-1], True) if len(m) > 1 else []
        if list(full[:len(pre)]) != list(pre):
            v_mis += 1
        v_max = max(v_max, len(full))
    rep["checks"]["text_rows_misaligned"] = t_mis
    rep["checks"]["vision_rows_misaligned"] = v_mis
    rep["checks"]["text_max_tokens"] = t_max
    rep["checks"]["vision_max_tokens"] = v_max
    assert t_mis == 0 and v_mis == 0, "template misalignment: text=%d vision=%d" % (t_mis, v_mis)

    del model_lm
    mx.clear_cache()
    log("model_assembly_report.json", rep)
    mark("A", rep["checks"])
    print("A done:", json.dumps(rep["checks"], default=str)[:600])


In [ ]:

# ---- Cell B: OCR smoke on the BASE 8-bit model (mlx-vlm) --------------------
if done("B"):
    print("B already done - skipping")
else:
    from PIL import Image
    import mlx_vlm
    from mlx_vlm import utils as vutils

    t0 = time.time()
    model, processor = vutils.load(MLX_DIR)
    load_s = time.time() - t0

    cases = [
        ("real", "data/processed/pages/homework_ava_week5_p1.png",
         "Transcribe the maths questions on this page exactly as written. Output only the transcription, one question per line.", 256),
        ("real", "data/processed/pages/homework_noah_week5_p1.png",
         "Transcribe the maths questions on this page exactly as written. Output only the transcription, one question per line.", 256),
    ]
    for i in (0, 2, 4):
        cases.append(("synth", "data/eval/handwriting_synth/hw_%04d.png" % i,
                      "Transcribe this maths question exactly as written. Output only the transcription.", 64))

    results = []
    for kind, path, instr, maxtok in cases:
        if not os.path.exists(path):
            results.append({"kind": kind, "path": path, "ok": False, "error": "missing file"})
            continue
        img = Image.open(path).convert("RGB")
        msgs = [{"role": "user", "content": [
            {"type": "image", "path": path},
            {"type": "text", "text": instr}]}]
        prompt = mlx_vlm.apply_chat_template(processor, model.config, msgs,
                                             add_generation_prompt=True, num_images=1)
        t1 = time.time()
        res = mlx_vlm.generate(model, processor, prompt, image=[img],
                               max_tokens=maxtok, temperature=0.2, top_p=0.95)
        txt = (res.text if hasattr(res, "text") else str(res)).strip()
        results.append({"kind": kind, "path": path, "ok": len(txt) > 0,
                        "chars": len(txt), "seconds": round(time.time() - t1, 1),
                        "output_head": txt[:160]})
        print("[ocr-smoke] %s %s -> %d chars in %.1fs" % (kind, path, len(txt), time.time()-t1))

    gate = all(r["ok"] for r in results)
    log("ocr_smoke_base.json", {"load_seconds": round(load_s, 1), "gate_all_nonempty": gate,
                                "results": results,
                                "note": "BASE model, pre-finetune. Baseline for M6 comparison."})
    del model, processor
    mx.clear_cache()
    assert gate, "OCR smoke failed on base model"
    mark("B", {"gate": gate, "seconds": [r["seconds"] for r in results]})


In [ ]:

# ---- Cell C: text-LoRA OVERFIT check (20 dialogues, prefix-expanded) --------
if done("C"):
    print("C already done - skipping")
else:
    import mlx_lm

    train_rows = [json.loads(l) for l in open("data/splits/train.jsonl")]
    for r in train_rows:
        r.setdefault("id", hashlib.sha256(json.dumps(r["messages"], sort_keys=True).encode()).hexdigest()[:16])

    overfit = expand(train_rows[:20])
    (OUT / "overfit_data").mkdir(exist_ok=True)
    with open(OUT / "overfit_data" / "train.jsonl", "w") as f:
        for r in overfit:
            f.write(json.dumps(r) + chr(10))
    with open(OUT / "overfit_data" / "valid.jsonl", "w") as f:
        for r in overfit[:2]:
            f.write(json.dumps(r) + chr(10))
    print("overfit data: %d prefix-expanded rows from 20 dialogues" % len(overfit))

    model, tokenizer = mlx_lm.load(MLX_DIR)
    t0 = time.time()
    cb = run_lora_training(
        model, tokenizer, overfit, overfit[:2],
        num_layers=16, lora_params={"rank": 16, "scale": 2.0, "dropout": 0.0},
        batch_size=1, iters=200, val_batches=2, learning_rate=5e-5, lr_schedule=None,
        max_seq_length=1024, grad_checkpoint=True, grad_accum=1,
        steps_per_report=25, steps_per_eval=50, save_every=200,
        adapter_path=str(OUT / "overfit_lora"), seed=0)
    dt = time.time() - t0

    first = cb.train[0]["loss"] if cb.train else None
    last = cb.train[-1]["loss"] if cb.train else None
    gate = (first is not None and last is not None and last < 0.9 * first)
    log("overfit_report.json", {"rows": len(overfit), "iters": 200, "seconds": round(dt, 1),
                                "first_loss": first, "last_loss": last, "gate": gate,
                                "train_curve": cb.train[::5], "val_curve": cb.val})
    del model, tokenizer
    mx.clear_cache()
    assert gate, "overfit gate failed: first=%s last=%s" % (first, last)
    mark("C", {"first": first, "last": last})
    print("C done: overfit loss %.4f -> %.4f (gate %s)" % (first, last, gate))


In [ ]:

# ---- Cell D: full text LoRA (all 706 dialogues, one epoch, grad accum) ------
if done("D"):
    print("D already done - skipping")
else:
    import mlx_lm

    train_rows = [json.loads(l) for l in open("data/splits/train.jsonl")]
    for r in train_rows:
        r.setdefault("id", hashlib.sha256(json.dumps(r["messages"], sort_keys=True).encode()).hexdigest()[:16])

    # dialogue-level split: last 20 dialogues -> validation (disjoint from train)
    n_valid = 20
    valid_dialogues = train_rows[-n_valid:]
    train_dialogues = train_rows[:-n_valid]

    train_exp = expand(train_dialogues)
    valid_exp = expand(valid_dialogues)
    (OUT / "text_lora_data").mkdir(exist_ok=True)
    with open(OUT / "text_lora_data" / "train.jsonl", "w") as f:
        for r in train_exp:
            f.write(json.dumps(r) + chr(10))
    with open(OUT / "text_lora_data" / "valid.jsonl", "w") as f:
        for r in valid_exp:
            f.write(json.dumps(r) + chr(10))
    print("text LoRA data: %d train rows (from %d dialogues) + %d valid rows (from %d)"
          % (len(train_exp), len(train_dialogues), len(valid_exp), n_valid))

    model, tokenizer = mlx_lm.load(MLX_DIR)
    iters = len(train_exp)          # one full epoch of the expanded rows
    updates = iters // 2            # grad_accum=2 -> optimizer updates
    lr_schedule = {"name": "cosine_decay", "arguments": [1e-5, updates, 1e-6],
                   "warmup": 50, "warmup_init": 1e-6}
    print("D: iters=%d (one epoch), grad_accum=2 -> %d updates, cosine to 1e-6" % (iters, updates))

    t0 = time.time()
    cb = run_lora_training(
        model, tokenizer, train_exp, valid_exp,
        num_layers=16, lora_params={"rank": 16, "scale": 2.0, "dropout": 0.0},
        batch_size=1, iters=iters, val_batches=10, learning_rate=1e-5, lr_schedule=lr_schedule,
        max_seq_length=2048, grad_checkpoint=True, grad_accum=2,
        steps_per_report=25, steps_per_eval=100, save_every=250,
        adapter_path=str(OUT / "text_lora"), seed=0)
    dt = time.time() - t0

    vfirst = cb.val[0]["loss"] if cb.val else None
    vlast = cb.val[-1]["loss"] if cb.val else None
    gate = (vfirst is not None and vlast is not None and vlast < vfirst)
    log("text_lora_report.json", {"train_rows": len(train_exp), "valid_rows": len(valid_exp),
                                  "iters": iters, "updates": updates, "seconds": round(dt, 1),
                                  "val_first": vfirst, "val_last": vlast, "gate": gate,
                                  "train_curve": cb.train[::10], "val_curve": cb.val})
    del model, tokenizer
    mx.clear_cache()
    assert gate, "text LoRA val loss did not decrease: first=%s last=%s" % (vfirst, vlast)
    mark("D", {"val_first": vfirst, "val_last": vlast})
    print("D done: val loss %.4f -> %.4f (gate %s)" % (vfirst, vlast, gate))


In [ ]:

# ---- Cell E: vision-tower fine-tune (60 rows, custom loop) ------------------
if done("E"):
    print("E already done - skipping")
else:
    from PIL import Image
    import mlx.nn as nn
    import mlx.optimizers as optim
    import mlx_vlm
    from mlx_vlm import utils as vutils
    from mlx_vlm.trainer.utils import grad_checkpoint, get_learning_rate

    t0 = time.time()
    model, processor = vutils.load(MLX_DIR)
    load_s = time.time() - t0
    model.train()   # pure-MLX delta-net path (Metal kernel has no VJP)

    rows = [json.loads(l) for l in open("data/splits/train_vision.jsonl")]
    prepared = []
    for r in rows:
        msgs = r["messages"]
        img_path = msgs[0]["content"][0]["path"]
        if not os.path.exists(img_path):
            continue
        img = Image.open(img_path).convert("RGB")
        p_full = mlx_vlm.apply_chat_template(processor, model.config, msgs, add_generation_prompt=False, num_images=1)
        p_pre  = mlx_vlm.apply_chat_template(processor, model.config, msgs[:1], add_generation_prompt=True, num_images=1)
        in_full = mlx_vlm.prepare_inputs(processor, images=[img], prompts=[p_full])
        in_pre  = mlx_vlm.prepare_inputs(processor, images=[img], prompts=[p_pre])
        fid = in_full["input_ids"]; pid = in_pre["input_ids"]
        off = pid.shape[1]
        assert bool((fid[:, :off] == pid).all()), "vision prefix misaligned"
        prepared.append({"input_ids": fid, "pixel_values": in_full.get("pixel_values"),
                         "grid": in_full.get("image_grid_thw"), "offset": off})
    print("prepared %d vision examples" % len(prepared))

    model.freeze()
    model.vision_tower.unfreeze()
    n_train = sum(p.size for _, p in tree_flatten(model.vision_tower.parameters()))
    n_total = sum(p.size for _, p in tree_flatten(model.parameters()))
    print("vision trainable %.1fM / %.1fM" % (n_train/1e6, n_total/1e6))
    assert n_train > 10_000_000, "vision tower too small: %d" % n_train

    grad_checkpoint(model.vision_tower.blocks[0])   # patches the block type for all blocks

    ITERS = min(len(prepared), 2000)
    opt = optim.Adam(learning_rate=1e-5)

    def loss_fn(model_, input_ids, pv, grid, offset):
        logits = model_(input_ids, pv, None, None, image_grid_thw=grid).logits
        logits = logits[:, :-1]
        targets = input_ids[:, 1:]
        steps = mx.arange(1, targets.shape[1] + 1)
        mask = mx.logical_and(steps >= offset, steps <= targets.shape[1])
        ce = nn.losses.cross_entropy(logits, targets) * mask
        return (ce.astype(mx.float32).sum() / mx.maximum(mask.sum(), 1))

    value_and_grad = nn.value_and_grad(model, loss_fn)

    losses_hist = []
    for it in range(1, ITERS + 1):
        opt.learning_rate = get_learning_rate(ITERS, it, 50, 1e-5, 1e-6)
        ex = prepared[(it - 1) % len(prepared)]
        lvalue, grad = value_and_grad(model, ex["input_ids"], ex["pixel_values"],
                                      ex["grid"], ex["offset"])
        opt.update(model, grad)
        mx.eval(model.parameters(), opt.state)
        lv = float(lvalue)   # mlx 0.32: mx.eval returns None
        losses_hist.append(lv)
        if it % 25 == 0 or it == 1:
            print("[vision] iter %d/%d loss %.4f lr %.2e" % (it, ITERS, lv, opt.learning_rate))

    vdir = OUT / "vision_tower"
    vdir.mkdir(parents=True, exist_ok=True)
    tower_keys = dict(tree_flatten(model.vision_tower.parameters()))
    mx.save_safetensors(str(vdir / "adapters.safetensors"), tower_keys)
    print("saved %d vision tower keys" % len(tower_keys))

    n = min(10, len(losses_hist))
    first = sum(losses_hist[:n]) / n; last = sum(losses_hist[-n:]) / n
    gate = last < first
    log("vision_tower_report.json", {"load_seconds": round(load_s, 1), "iters": ITERS,
        "first_loss": round(first, 4), "last_loss": round(last, 4), "gate_drops": gate,
        "loss_curve": [round(x, 4) for x in losses_hist[::10]]})
    del model, processor
    mx.clear_cache()
    assert gate, "vision tower loss did not drop: first=%s last=%s" % (first, last)
    mark("E", {"first": first, "last": last})
    print("E done: vision loss %.4f -> %.4f (gate %s)" % (first, last, gate))


In [ ]:

# ---- Cell F: merge LoRA + tower into standalone MLX dir + round-trip verify -
if done("F"):
    print("F already done - skipping")
else:
    import mlx_lm
    from mlx_lm.tuner.utils import load_adapters
    from mlx_lm.utils import dequantize_model
    from mlx_lm.tuner.lora import LoRALinear, LoRAEmbedding, LoRASwitchLinear
    from safetensors import safe_open

    lora_types = (LoRALinear, LoRAEmbedding, LoRASwitchLinear)

    # 1) language model: base 8-bit + trained text LoRA -> fused + dequantized
    model, tok = mlx_lm.load(MLX_DIR)
    load_adapters(model, str(OUT / "text_lora"))
    fused = [(k, m.fuse(dequantize=True)) for k, m in model.named_modules() if isinstance(m, lora_types)]
    print("fused %d LoRA modules" % len(fused))
    model.update_modules(tree_unflatten(fused))
    n_lora_left = sum(1 for k, m in model.named_modules() if isinstance(m, lora_types))
    assert n_lora_left == 0, "LoRA modules remain after fuse: %d" % n_lora_left
    dequantize_model(model)
    weights_lm = dict(tree_flatten(model.parameters()))
    print("language model: %d keys after merge+dequant" % len(weights_lm))
    assert len(weights_lm) == 426, "unexpected LM key count %d" % len(weights_lm)
    del model
    mx.clear_cache()

    # 2) vision tower: base 8-bit bf16 tower overlaid with trained keys (cell E)
    base_w = mx.load(os.path.join(MLX_DIR, "model.safetensors"))
    tower = {k: v for k, v in base_w.items() if k.startswith("vision_tower.")}
    del base_w
    mx.clear_cache()
    print("base vision tower: %d keys" % len(tower))
    # cell E saved tree_flatten(vision_tower) -> relative keys; restore the file prefix
    trained = {"vision_tower." + k: v for k, v in mx.load(str(OUT / "vision_tower" / "adapters.safetensors")).items()}
    print("trained vision keys: %d" % len(trained))
    tower.update(trained)

    # 3) combined standalone MLX dir (723 keys, all bf16/fp32)
    combined = {**weights_lm, **tower}
    print("combined: %d keys (LM %d + tower %d)" % (len(combined), len(weights_lm), len(tower)))
    assert len(combined) == 723, "unexpected combined key count %d" % len(combined)

    gdir = ROOT / GEETA_MLX_DIR
    gdir.mkdir(parents=True, exist_ok=True)
    mx.save_safetensors(str(gdir / "model.safetensors"), combined)
    index = {"metadata": {"total_size": sum(int(v.nbytes) for v in combined.values())},
             "weight_map": {k: "model.safetensors" for k in combined}}
    (gdir / "model.safetensors.index.json").write_text(json.dumps(index, indent=2))

    # config: pop quantization fields; keep token ids + text/vision config
    src_cfg = json.load(open(os.path.join(MLX_DIR, "config.json")))
    src_cfg.pop("quantization", None)
    src_cfg.pop("quantization_config", None)
    (gdir / "config.json").write_text(json.dumps(src_cfg, indent=2))

    # tokenizer + preprocessor files copied from the 8-bit dir
    for fn in os.listdir(MLX_DIR):
        if fn.endswith((".json", ".jinja")) and fn not in ("model.safetensors.index.json", "config.json"):
            try:
                import shutil
                shutil.copy2(os.path.join(MLX_DIR, fn), gdir / fn)
            except Exception as e:
                print("skip copy", fn, repr(e))

    # 4) round-trip verify: mlx_vlm.load the merged dir
    # free the in-memory combined dict (~9GB) first so the reload fits in unified memory
    n_combined = len(combined)
    del combined, weights_lm, tower, trained
    mx.clear_cache()
    import mlx_vlm
    from mlx_vlm import utils as vutils
    from PIL import Image
    m2, p2 = vutils.load(GEETA_MLX_DIR)

    # (a) real-page OCR
    rp = "data/processed/pages/homework_ava_week5_p1.png"
    rimg = Image.open(rp).convert("RGB")
    rmsgs = [{"role": "user", "content": [
        {"type": "image", "path": rp},
        {"type": "text", "text": "Transcribe the maths questions on this page exactly as written. Output only the transcription."}]}]
    rprompt = mlx_vlm.apply_chat_template(p2, m2.config, rmsgs, add_generation_prompt=True, num_images=1)
    ocr = (mlx_vlm.generate(m2, p2, rprompt, image=[rimg], max_tokens=160, temperature=0.2).text or "").strip()

    # (b) text chat (no image)
    tmsgs = [{"role": "user", "content": "What is 17 + 5? Just give the number."}]
    tprompt = mlx_vlm.apply_chat_template(p2, m2.config, tmsgs, add_generation_prompt=True, num_images=0)
    txt = (mlx_vlm.generate(m2, p2, tprompt, max_tokens=64, temperature=0.3).text or "").strip()

    # (c) tutor-style hint (never-reveal behaviour)
    hmsgs = [{"role": "user", "content": "I am stuck on 23 + 18. Tell me the answer please."}]
    hprompt = mlx_vlm.apply_chat_template(p2, m2.config, hmsgs, add_generation_prompt=True, num_images=0)
    hint = (mlx_vlm.generate(m2, p2, hprompt, max_tokens=128, temperature=0.5).text or "").strip()

    gate = len(ocr) > 0 and len(txt) > 0 and len(hint) > 0
    log("merge_verify.json", {"combined_keys": n_combined, "gate": gate,
                              "ocr_head": ocr[:160], "text_head": txt[:120], "hint_head": hint[:200]})
    del m2, p2
    mx.clear_cache()
    assert gate, "merged model round-trip failed"
    mark("F", {"combined_keys": n_combined})
    print("F done: merged GEETA MLX dir verified | ocr=%d text=%d hint=%d chars" % (len(ocr), len(txt), len(hint)))


In [ ]:

# ---- Cell G: HF dir assembly + GGUF (f16 + Q4_K_M) --------------------------
if done("G"):
    print("G already done - skipping")
else:
    import numpy as np
    from safetensors import safe_open

    # 1) load merged MLX weights
    mlx_w = dict(mx.load(os.path.join(GEETA_MLX_DIR, "model.safetensors")))
    print("merged MLX weights: %d keys" % len(mlx_w))

    def remap(k):
        if k.startswith("language_model.model."):
            return "model.language_model." + k[len("language_model.model."):]
        if k.startswith("vision_tower."):
            return "model.visual." + k[len("vision_tower."):]
        if k.startswith("language_model."):
            return "model." + k[len("language_model."):]
        return k

    hf_w = {}
    for k, v in mlx_w.items():
        nk = remap(k)
        a = np.asarray(v.astype(mx.float16))   # numpy has no bfloat16; convert in MLX first
        # shape fixes MLX -> HF
        if nk.endswith("conv1d.weight") and a.ndim == 3:
            a = np.transpose(a, (0, 2, 1))          # [C,4,1] -> [C,1,4]
        if nk.endswith("patch_embed.proj.weight") and a.ndim == 5:
            a = np.transpose(a, (0, 4, 1, 2, 3))    # [D,2,16,16,3] -> [D,3,2,16,16]
        hf_w[nk] = a.astype(np.float16)

    # R1 fix (session 12): mlx_vlm bakes convert_hf_to_gguf.py's "+1" RMSNorm
    # convention into language-model norm weights at load time (verified against
    # base checkpoint: input_layernorm mean 0.10 -> merged 1.10, exactly +1; same
    # for post_attention_layernorm, q_norm, k_norm, final norm). Qwen3_5TextModel's
    # modify_tensors (inherited from Qwen3NextModel) ADDS +1 AGAIN on conversion,
    # assuming raw unshifted input -> double-shift -> corrupted norms -> garbage
    # output. linear_attn.norm.weight and all vision tensors are correctly excluded
    # (verified unshifted) so must NOT be touched here.
    unshift_keys = [k for k in hf_w if k.startswith("model.language_model")
                     and k.endswith("norm.weight") and not k.endswith("linear_attn.norm.weight")]
    for k in unshift_keys:
        hf_w[k] = (hf_w[k].astype(np.float32) - 1.0).astype(np.float16)
    print("un-shifted %d language-model norm tensors by -1.0 (R1 fix)" % len(unshift_keys))
    assert len(unshift_keys) == 81, "expected 81 norm tensors, got %d - re-verify predicate" % len(unshift_keys)

    # 2) assert key set == HF index - mtp
    idx = json.load(open(os.path.join(HF_DIR, "model.safetensors.index.json")))
    hf_index = set(idx["weight_map"].keys())
    mtp = {k for k in hf_index if k.startswith("mtp.")}
    target = hf_index - mtp
    extra = set(hf_w) - target
    missing = target - set(hf_w)
    print("HF target (model-mtp): %d | produced: %d | extra: %d | missing: %d"
          % (len(target), len(hf_w), len(extra), len(missing)))
    assert not extra and not missing, "key mismatch extra=%s missing=%s" % (list(extra)[:5], list(missing)[:5])

    # 3) write HF dir
    hdir = ROOT / GEETA_HF_DIR
    hdir.mkdir(parents=True, exist_ok=True)
    mx.save_safetensors(str(hdir / "model.safetensors"), {k: mx.array(v) for k, v in hf_w.items()})
    (hdir / "model.safetensors.index.json").write_text(json.dumps(
        {"metadata": {"total_size": sum(int(v.nbytes) for v in hf_w.values())},
         "weight_map": {k: "model.safetensors" for k in hf_w}}, indent=2))
    import shutil
    shutil.copy2(os.path.join(HF_DIR, "config.json"), hdir / "config.json")
    for fn in os.listdir(HF_DIR):
        if fn.endswith((".json", ".jinja")) and "safetensors" not in fn and fn != "config.json":
            shutil.copy2(os.path.join(HF_DIR, fn), hdir / fn)
    print("HF dir written: %d keys" % len(hf_w))

    # 4) convert to GGUF (f16, no-mtp)
    import subprocess
    gguf_dir = ROOT / GEETA_GGUF_DIR
    gguf_dir.mkdir(parents=True, exist_ok=True)
    f16 = gguf_dir / "geeta-4b-f16.gguf"
    cmd = [sys.executable, str(ROOT / LLAMA_CPP / "convert_hf_to_gguf.py"),
           str(hdir), "--outtype", "f16", "--no-mtp", "--outfile", str(f16)]
    print("running:", " ".join(cmd))
    r = subprocess.run(cmd, cwd=LLAMA_CPP, capture_output=True, text=True)
    print(r.stdout[-800:])
    if r.returncode != 0:
        print("STDERR:", r.stderr[-1500:])
    assert f16.exists() and f16.stat().st_size > 1_000_000_000, "f16 GGUF not produced"
    print("f16 GGUF: %.2f GB" % (f16.stat().st_size / 2**30))

    # verify via gguf-py (vendored version: no context manager, use get_field/tensors)
    sys.path.insert(0, str(ROOT / LLAMA_CPP / "gguf-py"))
    import gguf
    g = gguf.GGUFReader(str(f16))
    af = g.get_field("general.architecture")
    arch = bytes(int(b) for b in af.parts[-1]).decode() if af is not None else None
    ntensors = len(g.tensors)
    del g
    print("gguf arch:", arch, "| tensors:", ntensors)

    # 5) quantize to Q4_K_M
    q4 = gguf_dir / "geeta-4b-q4_k_m.gguf"
    qcmd = [str(ROOT / LLAMA_CPP / "llama-quantize"), str(f16), str(q4), "Q4_K_M"]
    print("running:", " ".join(qcmd))
    rq = subprocess.run(qcmd, cwd=LLAMA_CPP, capture_output=True, text=True)
    print(rq.stdout[-500:])
    if rq.returncode != 0:
        print("STDERR:", rq.stderr[-1500:])
    assert q4.exists() and q4.stat().st_size > 100_000_000, "Q4 GGUF not produced"
    print("Q4_K_M GGUF: %.2f GB" % (q4.stat().st_size / 2**30))

    # 6) vision tower -> mmproj GGUF (llama.cpp mtmd vision path)
    mm = gguf_dir / "mmproj-geeta-4b-f16.gguf"
    mcmd = [sys.executable, str(ROOT / LLAMA_CPP / "convert_hf_to_gguf.py"),
            str(hdir), "--mmproj", "--outtype", "f16", "--outfile", str(mm)]
    print("running:", " ".join(mcmd))
    rm = subprocess.run(mcmd, cwd=LLAMA_CPP, capture_output=True, text=True)
    print(rm.stdout[-500:])
    if rm.returncode != 0:
        print("STDERR:", rm.stderr[-1500:])
    assert mm.exists() and mm.stat().st_size > 100_000_000, "mmproj GGUF not produced"
    print("mmproj GGUF: %.2f GB" % (mm.stat().st_size / 2**30))

    log("gguf_report.json", {"f16_gb": round(f16.stat().st_size / 2**30, 2),
                             "q4_gb": round(q4.stat().st_size / 2**30, 2),
                             "mmproj_gb": round(mm.stat().st_size / 2**30, 3),
                             "arch": str(arch), "hf_keys": len(hf_w)})
    mark("G", {"f16_gb": round(f16.stat().st_size / 2**30, 2), "q4_gb": round(q4.stat().st_size / 2**30, 2)})
    print("G done: GGUF f16 + Q4_K_M produced")


In [ ]:
# ---- Cell H: llama-server smoke (text + vision, TTFT) with MLX fallback ------
if done("H"):
    print("H already done - skipping")
else:
    import subprocess, base64, urllib.request, time as _time

    q4 = ROOT / GEETA_GGUF_DIR / "geeta-4b-q4_k_m.gguf"
    server_bin = str(ROOT / LLAMA_CPP / "llama-server")
    port = 8081

    def http_json(url, payload=None, method="GET"):
        data = json.dumps(payload).encode() if payload is not None else None
        req = urllib.request.Request(url, data=data, method=method,
                                     headers={"Content-Type": "application/json"})
        with urllib.request.urlopen(req, timeout=120) as resp:
            return json.loads(resp.read().decode())

    def http_json_retry(url, payload=None, method="GET", tries=6, delay=3):
        # R1 (session 12): as a safety net against any transient startup lag
        # (route mounting slightly behind /health), retry a couple times
        # before treating a request as a real failure.
        last = None
        for i in range(tries):
            try:
                return http_json(url, payload, method)
            except Exception as e:
                last = e
                print("  retry %d/%d for %s after %r" % (i + 1, tries, url, e))
                _time.sleep(delay)
        raise last

    def wait_health(base, tries=120):
        for _ in range(tries):
            try:
                http_json(base + "/health")
                return True
            except Exception:
                _time.sleep(2)
        return False

    result = {"backend": None, "ok": False}

    # --- primary: llama-server + mtmd ---
    proc = None
    try:
        mm = ROOT / GEETA_GGUF_DIR / "mmproj-geeta-4b-f16.gguf"
        cmd = [server_bin, "--model", str(q4), "--mmproj", str(mm),
               "--host", "127.0.0.1", "--port", str(port),
               "-c", "16384", "-ngl", "99"]
        print("starting:", " ".join(cmd))
        proc = subprocess.Popen(cmd, cwd=LLAMA_CPP, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        base = "http://127.0.0.1:%d" % port
        up = wait_health(base)
        print("llama-server health:", up)
        if up:
            result["backend"] = "llama.cpp"
            # /v1/models is informational only - never let it abort the smoke test.
            try:
                models = http_json_retry(base + "/v1/models")
                result["models"] = models.get("data", [])[:2]
            except Exception as e:
                result["models_error"] = repr(e)
                print("/v1/models check failed (non-fatal):", repr(e))

            # thinking OFF at inference to match training (tokenizer.apply_chat_template
            # was called with enable_thinking=False for every SFT row - see Cell A/C1).
            # Leaving this unset defaults the Qwen3.5 template to thinking ON, which is
            # off-distribution for the LoRA and burns max_tokens on <think> reasoning,
            # leaving message.content empty (R1, session 12 manual verification).
            ctk = {"enable_thinking": False}

            # (1) tutor text prompt
            # R1 root cause (session 12): the ORIGINAL code below called
            # http_json(url, payload) without method="POST" - http_json's default
            # is "GET", so every /v1/chat/completions call in this cell was sent
            # as an HTTP GET (with a JSON body) to a POST-only route, which
            # llama-server's router 404s. This - not a startup race - is why
            # every prior H run (sessions 11 and 12, run10/run11/run13 logs)
            # crashed here. Explicit method="POST" fixes it.
            t0 = _time.time()
            r1 = http_json_retry(base + "/v1/chat/completions", {
                "model": "geeta", "messages": [
                    {"role": "user", "content": "I am stuck on 23 + 18. Give me the first little hint, not the answer."}],
                "temperature": 0.7, "top_p": 0.95, "max_tokens": 128,
                "chat_template_kwargs": ctk}, method="POST")
            ttft1 = _time.time() - t0
            msg1 = r1.get("choices", [{}])[0].get("message", {})
            c1 = (msg1.get("content") or "").strip()
            result["text"] = {"ok": len(c1) > 0, "chars": len(c1), "ttft_s": round(ttft1, 2),
                              "head": c1[:200], "usage": r1.get("usage"),
                              "reasoning_chars": len((msg1.get("reasoning_content") or ""))}

            # (2) kid page as base64 data-URL image
            rp = "data/processed/pages/homework_ava_week5_p1.png"
            if os.path.exists(rp):
                b64 = base64.b64encode(open(rp, "rb").read()).decode()
                t0 = _time.time()
                r2 = http_json_retry(base + "/v1/chat/completions", {
                    "model": "geeta", "messages": [
                        {"role": "user", "content": [
                            {"type": "text", "text": "Transcribe the maths questions on this page exactly as written."},
                            {"type": "image_url", "image_url": {"url": "data:image/png;base64," + b64}}]}],
                    "temperature": 0.2, "max_tokens": 256,
                    "chat_template_kwargs": ctk}, method="POST")
                ttft2 = _time.time() - t0
                msg2 = r2.get("choices", [{}])[0].get("message", {})
                c2 = (msg2.get("content") or "").strip()
                result["vision"] = {"ok": len(c2) > 0, "chars": len(c2), "ttft_s": round(ttft2, 2),
                                    "head": c2[:200],
                                    "reasoning_chars": len((msg2.get("reasoning_content") or ""))}
            result["ok"] = result.get("text", {}).get("ok", False)
    finally:
        if proc is not None:
            proc.terminate()
            try:
                proc.wait(timeout=15)
            except Exception:
                proc.kill()

    # --- fallback: mlx_vlm OpenAI server ---
    if not result["ok"]:
        print("llama.cpp path failed; trying mlx_vlm server fallback")
        port2 = 8082
        proc2 = None
        try:
            cmd2 = [sys.executable, "-m", "mlx_vlm", "server", "--model", GEETA_MLX_DIR,
                    "--host", "127.0.0.1", "--port", str(port2)]
            print("starting:", " ".join(cmd2))
            proc2 = subprocess.Popen(cmd2, cwd=str(ROOT), stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
            base2 = "http://127.0.0.1:%d" % port2
            up = wait_health(base2)
            print("mlx_vlm server health:", up)
            if up:
                result["backend"] = "mlx_vlm"
                t0 = _time.time()
                r1 = http_json_retry(base2 + "/v1/chat/completions", {
                    "model": GEETA_MLX_DIR, "messages": [
                        {"role": "user", "content": "I am stuck on 23 + 18. Give me the first little hint, not the answer."}],
                    "temperature": 0.7, "max_tokens": 128}, method="POST")
                c1 = (r1.get("choices", [{}])[0].get("message", {}).get("content") or "").strip()
                result["text"] = {"ok": len(c1) > 0, "chars": len(c1), "ttft_s": round(_time.time()-t0, 2),
                                  "head": c1[:200]}
                result["ok"] = len(c1) > 0
        finally:
            if proc2 is not None:
                proc2.terminate()
                try:
                    proc2.wait(timeout=15)
                except Exception:
                    proc2.kill()

    log("serve_smoke.json", result)
    mark("H", {"backend": result.get("backend"), "ok": result.get("ok")})
    print("H done: backend=%s ok=%s" % (result.get("backend"), result.get("ok")))


In [ ]:

# ---- Cell I: summary --------------------------------------------------------
if done("I"):
    print("I already done - skipping")
else:
    def read(name):
        p = OUT / name
        return json.loads(p.read_text()) if p.exists() else None

    summary = {
        "A_assets": (read("model_assembly_report.json") or {}).get("checks"),
        "B_ocr_smoke": read("ocr_smoke_base.json"),
        "C_overfit": (read("overfit_report.json") or {}),
        "D_text_lora": (read("text_lora_report.json") or {}),
        "E_vision": (read("vision_tower_report.json") or {}),
        "F_merge": (read("merge_verify.json") or {}),
        "G_gguf": (read("gguf_report.json") or {}),
        "H_serve": read("serve_smoke.json"),
    }
    log("m4_summary.json", summary)

    print(chr(10) + "=" * 60)
    print("M4 SUMMARY")
    print("=" * 60)
    c = summary["A_assets"] or {}
    print("A assets   : keys=%s layers=%s text_mis=%s vision_mis=%s"
          % (c.get("dir_keys_total"), c.get("model_layers"), c.get("text_rows_misaligned"), c.get("vision_rows_misaligned")))
    b = summary["B_ocr_smoke"] or {}
    print("B ocr      : gate=%s load=%.1fs" % (b.get("gate_all_nonempty"), b.get("load_seconds", 0)))
    cc = summary["C_overfit"] or {}
    print("C overfit  : loss %.4f -> %.4f gate=%s" % (cc.get("first_loss", 0), cc.get("last_loss", 0), cc.get("gate")))
    d = summary["D_text_lora"] or {}
    print("D text lora: val %.4f -> %.4f gate=%s (%d rows)" % (d.get("val_first", 0), d.get("val_last", 0), d.get("gate"), d.get("train_rows", 0)))
    e = summary["E_vision"] or {}
    print("E vision   : loss %.4f -> %.4f gate=%s" % (e.get("first_loss", 0), e.get("last_loss", 0), e.get("gate_drops")))
    f = summary["F_merge"] or {}
    print("F merge    : keys=%s gate=%s" % (f.get("combined_keys"), f.get("gate")))
    g = summary["G_gguf"] or {}
    print("G gguf     : f16=%.2fGB q4=%.2fGB arch=%s" % (g.get("f16_gb", 0), g.get("q4_gb", 0), g.get("arch")))
    h = summary["H_serve"] or {}
    print("H serve    : backend=%s ok=%s" % (h.get("backend"), h.get("ok")))
    print("=" * 60)
    mark("I", {"done": True})
